In [ ]:
from pathlib import Path

import arcadia_pycolor as apc
import pandas as pd
from arcadia_microscopy_tools import MicroscopyImage
from tqdm.notebook import tqdm

font_dirpath = Path("/Users/ryanlane/Library/Fonts")
apc.mpl.setup(font_dirpath=str(font_dirpath))

In [ ]:
DATA_DIR = Path("../data/microscopy").resolve()

keys = ["strain", "bead_size_mm", "nd2_path"]
records = [
    ("WT", False, DATA_DIR / "WellB10_PointB10_0004_ChannelFITC BP_Seq0094.nd2"),
    ("WT", 3, DATA_DIR / "WellB06_PointB06_0002_ChannelFITC BP_Seq0128.nd2"),
    ("DEA2", False, DATA_DIR / "WellF10_PointF10_0007_ChannelFITC BP_Seq0277.nd2"),
    ("DEA2", 3, DATA_DIR / "WellF06_PointF06_0003_ChannelFITC BP_Seq0309.nd2"),
]

sample_metadata = pd.DataFrame(records, columns=keys)
sample_metadata

In [ ]:
from arcadia_microscopy_tools import ImageOperation, Pipeline
from arcadia_microscopy_tools.channels import DIC
from arcadia_microscopy_tools.masks import SegmentationMask
from arcadia_microscopy_tools.model import SegmentationModel
from arcadia_microscopy_tools.operations import rescale_by_percentile

pipeline = Pipeline([ImageOperation(rescale_by_percentile)], preserve_dtype=False)
model = SegmentationModel()

preprocessed_images = []
segmentation_masks = []
per_image_dfs = []
for i, row in tqdm(sample_metadata.iterrows(), total=len(sample_metadata)):
    strain = row["strain"]
    bead_size_mm = row["bead_size_mm"]
    nd2_path = row["nd2_path"]

    image = MicroscopyImage.from_nd2_path(nd2_path, channels=[DIC])  # was mistakenly saved as FITC
    pixel_size_um = image.metadata.instrument.channel_metadata_list[0].resolution.xy_step_um

    intensities = image.get_intensities_from_channel(DIC)
    preprocessed = pipeline(intensities)
    preprocessed_images.append(preprocessed)

    mask = model.segment(preprocessed)
    mask = SegmentationMask(mask)
    segmentation_masks.append(mask)

    cell_props = mask.convert_properties_to_microns(pixel_size_um)
    cell_props["strain"] = strain
    cell_props["bead_size_mm"] = bead_size_mm

    image_cell_df = pd.DataFrame.from_dict(cell_props)
    per_image_dfs.append(image_cell_df)

cell_measurements = pd.concat(per_image_dfs)
cell_measurements.groupby(["strain", "bead_size_mm"]).sample(3)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots()

palette = apc.palettes.primary.colors[:2]
sns.boxplot(
    data=cell_measurements,
    x="strain",
    y="axis_major_length_um",
    hue="bead_size_mm",
    palette=palette,
    ax=ax,
)
sns.stripplot(
    data=cell_measurements,
    x="strain",
    y="axis_major_length_um",
    hue="bead_size_mm",
    dodge=True,
    ax=ax,
    marker="o",
    alpha=0.6,
    palette=palette,
    linewidth=0.8,
    legend=False,
)

_ = ax.set_title("Major Axis Length by Strain and Bead Size")
_ = ax.set_ylabel("Major Axis Length (μm)")
_ = ax.set_xlabel("Strain")
_ = ax.legend(title="Bead Size (mm)")
_ = apc.mpl.style_plot(ax)
_ = sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(12, 12),
    layout="constrained",
)

for i, row in sample_metadata.iterrows():
    strain = row["strain"]
    bead_size_mm = row["bead_size_mm"]
    ax = axes.flatten()[i]

    preprocessed = preprocessed_images[i]
    ax.imshow(preprocessed, cmap="gray")

    mask = segmentation_masks[i]
    for coords_yx in mask.cell_outlines:
        ax.plot(coords_yx[:, 1], coords_yx[:, 0], "#ffff00", lw=1)

    bead_str = f"{bead_size_mm} mm beads" if bead_size_mm else "No beads"
    title = f"{strain} | {bead_str}\n{mask.num_cells} cells found"
    ax.set_title(title)
    ax.axis("off")